In [13]:
import pandas as pd

data = pd.read_csv('/scratch2/mlavechin/phorec/phonetically_transcribed_pairs/utterances.csv', sep='\t')
data

/tmp/ipykernel_3617310/3810971317.py:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/scratch2/mlavechin/phorec/phonetically_transcribed_pairs/utterances.csv', sep='\t')


,speaker_type,speaker_role,speaker,content,phones,onset,offset,transcript_path,audio_path,age,...,gender,ses,speaker_group,file_design,file_group,file_activity,corpus,database,duration,age_months
0,KCHI,Target_Child,CHI,olá .,ɐˈjɐ,1640210.0,1643210.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,1;05.16,...,female,NaN,MC,long,TD,toyplay,CCF,phon,3000.0,17.53
1,KCHI,Target_Child,CHI,olá .,ˈ*aː,1647088.0,1650088.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,1;05.16,...,female,NaN,MC,long,TD,toyplay,CCF,phon,3000.0,17.53
2,KCHI,Target_Child,CHI,olá .,ɐˈlaː,1652698.0,1655698.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,1;05.16,...,female,NaN,MC,long,TD,toyplay,CCF,phon,3000.0,17.53
3,KCHI,Target_Child,CHI,olá .,ɐˈna,1655699.0,1657668.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,1;05.16,...,female,NaN,MC,long,TD,toyplay,CCF,phon,1969.0,17.53
4,KCHI,Target_Child,CHI,olá .,ɐˈ*a,1736073.0,1739073.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,1;05.16,...,female,NaN,MC,long,TD,toyplay,CCF,phon,3000.0,17.53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
840969,KCHI,Target_Child,CHI,c'est le petit train !,se lə pti tʁɛ̃,3479538.0,3481374.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,4;03.26,...,male,NaN,NaN,long,TD,toyplay,Yamaguchi,phon,1836.0,51.87
840970,KCHI,Target_Child,CHI,oh elle est allée plus vite celle-là !,o ɛl e ale py vit sɛlla,3481375.0,3484881.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,4;03.26,...,male,NaN,NaN,long,TD,toyplay,Yamaguchi,phon,3506.0,51.87
840971,KCHI,Target_Child,CHI,encore !,ɔ̃kʲɔʁ,3484882.0,3486686.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,4;03.26,...,male,NaN,NaN,long,TD,toyplay,Yamaguchi,phon,1804.0,51.87
840972,KCHI,Target_Child,CHI,xxx .,*,3486687.0,3492616.0,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,/scratch1/data/raw_data/CLEAR_ASR/IN_PREP/down...,4;03.26,...,male,NaN,NaN,long,TD,toyplay,Yamaguchi,phon,5929.0,51.87


In [2]:
# Draw phonetic inventory used in our data
# So that we provide a mapping for all categories
import panphon as panphon
from tqdm.notebook import tqdm

ft = panphon.FeatureTable()
all_categories = set()
for utterance in tqdm(data['phones']):
    if not pd.isna(utterance):
        phones = ft.ipa_segs(utterance)
        all_categories.update(phones)
print(f"Found {len(all_categories)} categories.")

  0%|          | 0/840974 [00:00<?, ?it/s]

Found 965 categories.


In [3]:
import sys
sys.path.append('/scratch2/mlavechin/phorec')
from ipa.categories import vowels, consonants
import panphon.distance as pdist
my_inventory = vowels | consonants

def simplify_sound(complex_sound, target_inventory):
    if complex_sound in target_inventory:
        return complex_sound
        
    feature_table = panphon.FeatureTable()
    distance_calculator = pdist.Distance()

    distances = {}
    for sound in target_inventory:
        distance = distance_calculator.feature_edit_distance(complex_sound, sound)
        distances[sound] = distance

    closest_sound = min(distances, key=distances.get)
    return closest_sound

simplifier = {}
for category in tqdm(all_categories):
    new_category = simplify_sound(category, my_inventory)
    simplifier[category] = new_category

  0%|          | 0/965 [00:00<?, ?it/s]

In [6]:
# You can copy/paste this phorec/ipa/mapping.py
print("Mapping")
for k, v in simplifier.items():
    print(f"'{k}': '{v}',")

Mapping
'pʷ': 'p',
'w̥': 'w',
'ɪ̤ː': 'ɪ',
'pʰʷ': 'p',
'ɶ̥': 'ø',
'ɸ̞': 'f',
'l̰': 'l',
'ɘ̘': 'ɐ',
'ð̝': 'ð',
's̪ʷ': 'θ',
'zːʲ': 'z',
'k': 'k',
'kʲʰ': 'k',
'pˠ': 'p',
'sʼ': 's',
'ɦʷː': 'w',
'ɑ̥': 'ɑ',
'lʷ': 'ɹ',
'nʷ': 'n',
'kˀ': 'k',
'tⁿ': 't',
'ʙʷ': 'ɥ',
'fː': 'f',
'ɨ̰': 'ɨ',
'ɯ̥': 'ə',
'j̞': 'j',
'mʷ': 'm',
'ɾʲ': 'r',
'æ̞': 'æ',
'i̤': 'i',
'ø̃': 'ø',
'tʷʰ': 't',
'ɲ̩': 'ɲ',
'ɻ̝': 'j',
'ɪ̥ː': 'ɪ',
'ɣː': 'ɡ',
'ʎː': 'ʎ',
'ʏ̈': 'ʏ',
'a̤': 'a',
'ʃ̠': 'ʃ',
'ɔ̆': 'ɔ',
'ɒ': 'ɒ',
'ɬʲ': 'r',
'ʊ̤ː': 'ʊ',
'o̰ː': 'o',
'ɐ̤ː': 'ɐ',
'ʎ': 'ʎ',
'õ̰': 'õ',
'l̻': 'l',
'æ̙': 'ɛ',
'i̥ː': 'i',
'ʝː': 'ç',
'sʷ': 's',
'ʃ̟': 'ʃ',
'nːʲ': 'n',
'j̩': 'i',
'z̪ʷ': 'ʒ',
'n̥ː': 'n',
'æ̯': 'æ',
'ʊ̠': 'ʊ',
'd̼': 'd',
'ð̤': 'ð',
'zː': 'z',
'ʒ̩': 'ʒ',
'e̝': 'ɐ',
't': 't',
'ʌ̰ː': 'ʌ',
'ɑ̃': 'ɑ̃',
'ɶ': 'ø',
'bⁿ': 'm',
'ɖ': 'd',
'çː': 'ç',
't̺': 't',
'ʑʷ': 'ʒ',
'ʃˠ': 'ʃ',
'ɘ̥': 'ɐ',
'o̥': 'o',
'ɨ̃': 'ɨ',
's̺': 's',
'z̃': 'z',
'tʷ': 't',
'ɹʷ': 'ɹ',
'ɑ̤': 'ɑ',
'ɻ': 'j',
'ø̤': 'ø',
'eː': 'ɐ',
'ɸ': 'f',
'œ̰': 'œ',
'ʀ̞': 'h',
'v

In [25]:
output_categories = set(simplifier.values())
print(f'Found {len(output_categories)} categories in the target inventory')
for c in output_categories:
    print(c)
missing_categories = my_inventory - output_categories
print(f'Categories in the target inventory not found in the data:', missing_categories)

Found 59 categories in the target inventory
ʌ
ʎ
ʁ
j
ɡ
ø
ʏ
ɪ
ĩ
ʊ
ŋ
ɨ
ɐ
a
b
ɐ̃
ɑ
ç
e
d
u
t
f
ɜ
õ
ɔ
ũ
h
ɒ
ɥ
n
ɑ̃
p
o
i
ɲ
x
y
r
ʒ
ʃ
ɔ̃
œ̃
ɛ
ð
w
l
z
s
θ
m
ɛ̃
ɾ
œ
k
v
ɹ
ə
æ
Categories in the target inventory not found in the data: set()


In [23]:

forbidden_characters = {
    'X',
    'V',
    'C',
    'xx',
}


def phone_mapper(utterance):
    if any(forbidden in utterance for forbidden in forbidden_characters):
        return []
    ft = panphon.FeatureTable()
    phones = ft.ipa_segs(utterance)
    phones = [simplifier[sound] for sound in phones]
    return phones
    
utterance = data.loc[0,'phones']
for idx, row in data.iterrows():
    utterance = row['phones']
    transcript_path = row['transcript_path']
    phones = phone_mapper(utterance)
    if 'e' in utterance:
        print(utterance, '-->', phones)
        break


aˈpe --> ['a', 'p', 'e']


In [11]:
import panphon as panphon
import sys
sys.path.append('/scratch2/mlavechin/phorec')
from ipa.mapping import simplifier

def simplify_phones(phones, ft, simplifier):
    if pd.isna(phones):
        return ''
    phones = ft.ipa_segs(phones)
    phones = [simplifier[sound] for sound in phones]
    return ' '.join(phones)

ft = panphon.FeatureTable()
data['simplified_phones'] = data['phones'].apply(lambda x: simplify_phones(x, ft, simplifier))

0                                     ɐ j ɐ
1                                         a
2                                     ɐ l a
3                                     ɐ n a
4                                       ɐ a
                        ...                
840969                 s e l ə p t i t ʁ ɛ̃
840970    o ɛ l e a l e p y v i t s ɛ l l a
840971                             ɔ̃ k ɔ ʁ
840972                                     
840973                          a n a n ə a
Name: simplified_phones, Length: 840974, dtype: object
